In [1]:
import cv2
import matplotlib.pyplot as plt

# Cargar una imagen desde URL o archivo
url = "https://upload.wikimedia.org/wikipedia/commons/7/70/Example.png"
import urllib.request
import numpy as np
resp = urllib.request.urlopen(url)
image = np.asarray(bytearray(resp.read()), dtype="uint8")
image = cv2.imdecode(image, cv2.IMREAD_COLOR)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.imshow(image)
plt.title("Imagen original")
plt.axis("off")
plt.show()


HTTPError: HTTP Error 403: Forbidden

In [2]:
from skimage.filters import threshold_otsu

gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
thresh_val = threshold_otsu(gray)
binary = gray > thresh_val

plt.imshow(binary, cmap='gray')
plt.title("Segmentación por Umbral (Otsu)")
plt.axis("off")
plt.show()


NameError: name 'image' is not defined

In [ ]:
Z = image.reshape((-1,3))
Z = np.float32(Z)

# KMeans con 3 clusters
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
K = 3
_, labels, centers = cv2.kmeans(Z, K, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
centers = np.uint8(centers)
segmented = centers[labels.flatten()]
segmented = segmented.reshape(image.shape)

plt.imshow(segmented)
plt.title("Segmentación con K-means (K=3)")
plt.axis("off")
plt.show()

In [ ]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = maskrcnn_resnet50_fpn(pretrained=True)
model.eval().to(device)

transform = transforms.Compose([transforms.ToTensor()])
img_tensor = transform(image).to(device)

with torch.no_grad():
    pred = model([img_tensor])

# Visualizar primeras máscaras detectadas
masks = pred[0]['masks'][:3]
labels = pred[0]['labels'][:3]

fig, axes = plt.subplots(1, 3, figsize=(12,4))
for i, ax in enumerate(axes):
    mask = masks[i, 0].mul(255).byte().cpu().numpy()
    ax.imshow(mask, cmap='gray')
    ax.set_title(f"Objeto {labels[i].item()}")
    ax.axis("off")
plt.show()
